# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [13]:
import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY").strip()
os.environ["GOOGLE_API_KEY"] = api_key

print("API key loaded successfully.")
print("Key length:", len(api_key))

API key loaded successfully.
Key length: 53


In [2]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [3]:
!pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.

In [4]:
# User input needed

DOCUMENTS = {
    "python_programming": """
    Python is a high-level programming language known for its simple syntax.
    It supports object-oriented, procedural, and functional programming.
    Python uses variables, loops, functions, lists, tuples, and dictionaries.
    It is widely used in web development, automation, artificial intelligence, and data science.
    """,

    "data_structures": """
    Data structures are used to organize and store data efficiently.
    Arrays store elements in a continuous memory location.
    Stacks follow the Last In First Out principle.
    Queues follow the First In First Out principle.
    Linked lists consist of nodes connected using pointers.
    """,

    "artificial_intelligence": """
    Artificial Intelligence enables machines to perform tasks that normally require human intelligence.
    Machine Learning is a branch of AI that learns patterns from data.
    Deep Learning uses neural networks with multiple layers.
    Generative AI can create new content such as text, images, audio, and code.
    AI is used in healthcare, education, transportation, and many other fields.
    """,

    "database_management": """
    A database is an organized collection of related information.
    Database Management Systems are used to store, retrieve, and manage data.
    SQL is a language used to communicate with relational databases.
    Primary keys uniquely identify records in a database table.
    Foreign keys establish relationships between different database tables.
    """
}

In [5]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("Python programming"))


[Source: python_programming]

    Python is a high-level programming language known for its simple syntax.
    It supports object-oriented, procedural, and functional programming.
    Python uses variables, loops, functions, lists, tuples, and dictionaries.
    It is widely used in web development, automation, artificial intelligence, and data science.
    


In [7]:
model="gemini-2.5-flash"

In [24]:
# @title Ask Gemini to search for something

import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.8-flash",
    tools=[search_documents],
    system_instruction=(
        """
        You are a Computer Science Study Assistant.

        You help students understand topics using the
        provided Computer Science documents.

        When you need specific information, use the
        search_documents tool.

        Only answer based on information found in the documents.
        If the documents do not contain the answer,
        clearly say that the information is not available.

        Give clear, simple and accurate explanations.
        """
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.8-flash


/tmp/ipykernel_9156/2391672450.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [22]:
import os
import json
from google import genai
from google.genai import types

# Connect to Gemini with a 120-second timeout
client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"].strip(),
    http_options=types.HttpOptions(timeout=120000)
)

MODEL = "gemini-3.8-flash"

# Define the document search tool
search_tool = {
    "type": "function",
    "name": "search_documents",
    "description": "Search the document collection for relevant information.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Topic to search in the documents."
            }
        },
        "required": ["query"]
    }
}


# ReAct agent loop
def run_agent(question, max_steps=3):

    print("Question:", question, flush=True)
    print("-" * 50, flush=True)

    interaction = None
    function_results = None

    for step_number in range(1, max_steps + 1):

        print(
            f"\nStep {step_number}: Contacting Gemini...",
            flush=True
        )

        try:
            if step_number == 1:
                interaction = client.interactions.create(
                    model=MODEL,
                    input=question,
                    tools=[search_tool]
                )
            else:
                interaction = client.interactions.create(
                    model=MODEL,
                    previous_interaction_id=interaction.id,
                    input=function_results,
                    tools=[search_tool]
                )

        except Exception as e:
            print("\nGemini request failed:", str(e), flush=True)
            return "Agent stopped because of an API error."

        print("Gemini responded.", flush=True)

        calls = [
            item for item in (interaction.steps or [])
            if item.type == "function_call"
        ]

        # If no tool call remains, show the final answer
        if not calls:
            print("\nFinal Answer:", flush=True)
            print(interaction.output_text, flush=True)
            return interaction.output_text

        function_results = []

        # Execute document searches
        for call in calls:

            print(
                f"\nCalling tool: {call.name}",
                flush=True
            )

            if call.name == "search_documents":
                result = search_documents(**call.arguments)
            else:
                result = "Unknown tool."

            print("Document search completed.", flush=True)

            function_results.append({
                "type": "function_result",
                "name": call.name,
                "call_id": call.id,
                "result": [
                    {
                        "type": "text",
                        "text": json.dumps({"result": result})
                    }
                ]
            })

    print("\nMaximum agent steps reached.", flush=True)
    return "Agent stopped."


# Test the ReAct agent
run_agent(
    "Explain the main features of Python programming. Use the document search tool."
)

Question: Explain the main features of Python programming. Use the document search tool.
--------------------------------------------------

Step 1: Contacting Gemini...
Gemini responded.

Calling tool: search_documents
Document search completed.

Step 2: Contacting Gemini...
Gemini responded.

Final Answer:
Based on the document, the main features of Python programming include:

* **High-Level Language & Simple Syntax:** Python is designed to be readable and easy to learn with straightforward syntax.
* **Multiple Programming Paradigms:** It supports diverse programming styles, including:
  * Object-oriented programming
  * Procedural programming
  * Functional programming
* **Core Built-in Constructs & Data Structures:** Python utilizes fundamental programming constructs such as variables, loops, and functions, alongside built-in data structures like lists, tuples, and dictionaries.
* **Broad Application Domains:** It is widely used across various fields, including web development, au

'Based on the document, the main features of Python programming include:\n\n* **High-Level Language & Simple Syntax:** Python is designed to be readable and easy to learn with straightforward syntax.\n* **Multiple Programming Paradigms:** It supports diverse programming styles, including:\n  * Object-oriented programming\n  * Procedural programming\n  * Functional programming\n* **Core Built-in Constructs & Data Structures:** Python utilizes fundamental programming constructs such as variables, loops, and functions, alongside built-in data structures like lists, tuples, and dictionaries.\n* **Broad Application Domains:** It is widely used across various fields, including web development, automation, artificial intelligence (AI), and data science.'

In [23]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Add a question here.",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: Add a question here.
--------------------------------------------------

Step 1: Contacting Gemini...
Gemini responded.

Final Answer:
It looks like you might have accidentally left placeholder text! Please feel free to ask your question, and I'll be happy to help.

Question: How many ghosts eat carrots everyday?
--------------------------------------------------

Step 1: Contacting Gemini...
Gemini responded.

Calling tool: search_documents
Document search completed.

Step 2: Contacting Gemini...
Gemini responded.

Final Answer:
Based on folklore, mythology, and reality, **none** (zero). Ghosts are considered non-corporeal entities without physical bodies or digestive systems, so they cannot eat food.

Question: What is the most popular food item in Canada?
--------------------------------------------------

Step 1: Contacting Gemini...
Gemini responded.

Calling tool: search_documents
Document search completed.

Step 2: Contacting Gemini...
Gemini responded.

Final Answer:

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)

In [26]:
# Questions for testing the ReAct agent

questions = [
    "What are the main features of Python programming?",
    "Explain the principles of stacks and queues.",
    "What is Artificial Intelligence and what are its applications?",
    "What is a Database Management System and what is SQL used for?"
]

# Check whether at least four questions are added
assert len(questions) >= 4

# Check that the placeholder question has been removed
assert "Add a question here." not in questions

print("All checks passed!")
print("Total questions:", len(questions))

# Display all questions
for i, question in enumerate(questions, start=1):
    print(f"{i}. {question}")

All checks passed!
Total questions: 4
1. What are the main features of Python programming?
2. Explain the principles of stacks and queues.
3. What is Artificial Intelligence and what are its applications?
4. What is a Database Management System and what is SQL used for?


In [18]:
from google import genai
import os

api_key = os.environ["GOOGLE_API_KEY"].strip()

client = genai.Client(api_key=api_key)

response = client.interactions.create(
    model="gemini-3.8-flash",
    input="Say hello"
)

print(response.output_text)

Hello! How can I help you today?
